### Import 안될 때 해결법
- __pycache__속 파일 삭제
- 하단 코드에서 내가 원하는 파일의 이름을 Basic_operations 대신 입력 후 실행

In [1]:
import importlib
import euclidean_ct
importlib.reload(euclidean_ct)

<module 'euclidean_ct' from '/home/junhyung/study/Data-Analysis-with-CKKS/Cluster/DBSCAN_CKKS/euclidean_ct.py'>

### Instance 생성

In [2]:
from desilofhe import Engine
from EncryptModule import CKKSEncryptor
encryptor = CKKSEncryptor()

engine = encryptor.get_engine()
secret_key = encryptor.get_secret_key()
public_key = encryptor.get_public_key()
relinearization_key = encryptor.get_relinearization_key()
conjugation_key = encryptor.get_conjugation_key()
bootstrap_key = encryptor.get_bootstrap_key()
rotation_key = encryptor.get_rotation_key()

### DBSCAN 함수

1. neighbor 판별 함수(_eps_neighborhood)
2. neighbor 리스트 출력함수(_region_query)
3. 클러스터 생성 & 확장 함수(_expand_cluster)
4. main 함수(run_dbscan)

1. Neighbor 판별 함수(_eps_neighborhood)
    - Neighbor과의 거리가 eps 이상인지 이하인지 판별하는 함수
    - 모든 다른 점과의 거리 진행
        - 이를 한번에 진행하여 eps 이하이면 진행

In [ ]:
# _eps_neighborhood
"""
comp 후 결과 -> and 대신 evalsign 써보기
"""
from Basic_operations import CKKS_comp
import numpy as np

def encrypted_euclidean_2d(engine, p_enc, q_enc, rotation_key, relinearization_key):
    """2차원 점들 간 거리 계산 시 사용
    Ex. p_enc = (x1,y1,x2,y2,x3,y3....)
    q_enc = (x4,y4,x5,y5,x6,y6....)"""
    # 암호문 차이 계산
    diff = engine.subtract(p_enc, q_enc)
    
    # 각 슬롯별 제곱 (x_diff^2, y_diff^2, ...)
    diff_squared = engine.square(diff, relinearization_key)
    
    # rotation 1 슬롯 (y_diff^2, x_next_diff^2, ...)
    rotated = engine.rotate(diff_squared, rotation_key, 1)
    
    # 두 벡터를 더해 각 점별 거리 제곱 계산 (x_diff^2 + y_diff^2)
    distance_squared_enc = engine.add(diff_squared, rotated)
    return distance_squared_enc

def _eps_neighborhood(engine, p_enc, q_enc, eps, d_a, d_b, t, m, relinearization_key,
                       conjugation_key, bootstrap_key, max_diff_sq_inv):
    #제곱거리계산(홀수번째 index(0-base)에 저장되어있음)
    dist = encrypted_euclidean_2d(engine, p_enc, q_enc, rotation_key, relinearization_key)
    
    # 암호문에 마스크 곱셈 (slot-wise masking)
    
    dist_masked_temp2 = engine.multiply(dist,max_diff_sq_inv)
    dist_masked = engine.add(dist_masked_temp2,0.5)
    dist_eps = engine.add(eps, 0.5)
    dist_masked = engine.bootstrap(dist_masked, relinearization_key, conjugation_key, bootstrap_key)
    dist_comp = CKKS_comp(engine, dist_masked, dist_eps, d_a, d_b, t, m, relinearization_key, conjugation_key, bootstrap_key) # 1/2 <= x < 3/2
    return dist_comp

def main():
    data   = [1,2,7,9,5,6,3,8,4,10] 
    data_1 = [3,8,4,10,1,2,7,9,5,6]
    max1 = max(data)
    min1 = min(data)
    max2 = max(data_1)
    min2 = max(data_1)
    if max1 > max2:
        max_temp = max1
    else:
        max_temp = max2
    if min1 > min2:
        min_temp = min2
    else:
        min_temp = min1
    max_diff_square = float((max_temp - min_temp)**2)
    max_diff_sq_inv = np.reciprocal(max_diff_square)
    scale_down_factor = 10
    eps_raw = [20]*len(data) # eps: 2
    # 1/2~3/2 범위를 맞추기 위한 예시 스케일링(0.5 + x/10)
    val1 = [x / scale_down_factor for x in data]
    val2 = [x / scale_down_factor for x in data_1]
    eps_scaled = [( x / scale_down_factor)**2 for x in eps_raw]
    enc_eps = engine.encrypt(eps_scaled, public_key)
    enc1 = engine.encrypt(val1, public_key)
    enc2 = engine.encrypt(val2, public_key)
    comp_out = _eps_neighborhood(engine, enc1, enc2, enc_eps, 5, 6, 13, 8,
                                relinearization_key, conjugation_key, bootstrap_key, max_diff_sq_inv)
    dec = engine.decrypt(comp_out, secret_key)  # 복호화 후 전체 슬롯 반환
    dec_real = [z.real for z in dec]

    # 홀수 슬롯만 골라서 출력
    for i in range(len(data)-1):
        value = dec_real[2*i+1]  # 짝수 슬롯 인덱스만 접근
        print(f"\n=== {i}번째 비교: ({(data[2*i] - data_1[2*i])**2+(data[2*i+1] - data_1[2*i+1])**2}) vs. {eps_raw[0]} ===")
        if value > 0.5:
            print(f"{(data[2*i] - data_1[2*i])**2+(data[2*i+1] - data_1[2*i+1])**2} > {eps_raw[0]**2}")
        else:
            print(f"{(data[2*i] - data_1[2*i])**2+(data[2*i+1] - data_1[2*i+1])**2} <= {eps_raw[0]**2}")
        print(f"comp_out ≈ {value:.6f}")

    return 0

if __name__ == '__main__':
    main()



=== 0번째 비교: (40) vs. 20 ===
40 > 400
comp_out ≈ 225634662288846780684175468543344640.000000

=== 1번째 비교: (10) vs. 20 ===
10 > 400
comp_out ≈ 688906932332448260151727607345315840.000000

=== 2번째 비교: (32) vs. 20 ===
32 > 400
comp_out ≈ 549311635824583281830279952835018752.000000

=== 3번째 비교: (17) vs. 20 ===
17 > 400
comp_out ≈ 625040524096720933442937097704964096.000000

=== 4번째 비교: (17) vs. 20 ===
17 > 400
comp_out ≈ 2502731076211253490618530280222228480.000000


IndexError: list index out of range

In [2]:
#데이터 받고 실행까지 진행

def prepare_points_vector(points):
    # points: [(x1,y1), (x2,y2), ..., (x10,y10)]
    flattened = []
    for point in points:
        flattened.extend(point)
    return flattened

def replicate_point(point, num_points):
    # point: (x,y)
    return point * num_points  # (x,y,x,y,...)

def neighborhood_simd(engine, points_enc, target_point_enc, eps, other_params):
    # 빼기, 제곱, rotation+add로 거리 계산
    diff = engine.subtract(points_enc, target_point_enc)
    diff_squared = engine.square(diff)
    dist_squared = sum_rotation_add(diff_squared, engine, dimension=2)
    comp_result = CKKS_comp(engine, dist_squared, eps, other_params)
    return comp_result

# sum_rotation_add 예: 2차원 거리 합 구하기
def sum_rotation_add(enc_vector, engine, dimension):
    result = enc_vector
    step = 1
    while step < dimension:
        rotated = engine.rotate(result, step)
        result = engine.add(result, rotated)
        step *= 2
    return result


In [20]:
import numpy as np

a = 11
b = float(a-1)
print(b)

print(np.reciprocal(81.0))

10.0
0.012345679012345678
